[Time Series Features](https://otexts.com/fpppy/04-features.html)

[Autocovariance, Autocorrelation](https://github.com/ajitsingh98/Time-Series-Analysis-and-Forecasting-with-Python/blob/master/Time_Series_Forecasting_Traditional_Methods.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JayitaCh/Time-Series-Analysis/blob/python/Ch4_4TS_statistics.ipynb)

# **Time Series Statistics**

## **Detecting Unusual or Anomalous Time Series**

In [7]:
!git clone -b python https://github.com/JayitaCh/Time-Series-Analysis.git

Cloning into 'Time-Series-Analysis'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 158 (delta 59), reused 85 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 34.48 MiB | 10.88 MiB/s, done.
Resolving deltas: 100% (59/59), done.


In [8]:
%cd Time-Series-Analysis

/content/Time-Series-Analysis


#### Load Libraries

In [26]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
np.set_printoptions(suppress=True)
np.random.seed(1)
import random
random.seed(1)
pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_style("whitegrid")
import matplotlib.pyplot as plt
plt.style.use("ggplot")

import tsfeatures as tsf

from IPython.display import display

When we collect large amount of data, it is quite natural to obtain some ambiguous outputs at different instances of time. It is therefore important to detect such **unusual or anomalous time series**.

Such detection of unusual behaviour can be carried out by measuring different statistics. Python library `tsfeature` helps to compute a vector of features on each time series, measuring different characteristic-features of the series. The features may include **lag correlation**, the **strength of seasonality**, **spectral entropy**, etc.

<div align="center">

| Feature                 | Function                    | Description                                             |
| ----------------------- | --------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Mean**                | `intervals`                 | Computes the mean of intervals of positive values.                                                                                                                                                             |
| **SD**                  | `intervals`                 | Computes the SD of intervals of positive values.                                                                                                                                                               |
| **ACF1**                | `acf_features`              | **Autocorrelation:** Computes the autocorrelation function for the original, first-differenced, and twice-differenced series.<br> Also gives the first ACF coefficient and sum of squares of the first 10 coefficients.                 |
| **PACF1**               | `pacf_features`             | **Partial Autocorrelation:** Computes the partial autocorrelation function for the original, first-differenced, and twice-differenced series.<br> Also gives the first PACF coefficient and sum of squares of the first 10 coefficients.        |
| **Trend**               | `stl_features`              | Obtained through STL decomposition. Trend strength is measured as<br> $$ trend = 1 - \frac{Var(e_t)}{Var(f_t + e_t)} $$                    |
| **Linearity**           | `nonlinearity`              | Measures the strength of linearity/non-linearity using the SSE ratio from nonlinear and linear autoregression.<br> Large values indicate non-linearity; values near 0 indicate linearity.<br> $$ X_2 = Tlog(\frac{SSE_1}{SSE_0}) $$                |
| **Curvature**           | `stl_features`              | Measures the strength of curvature using trend and seasonality measures from STL decomposition,<br> primarily coefficients from an orthogonal quadratic regression.                                                |
| **Season**              | `stl_features`              | Measures the strength of seasonality.<br> Values below 0 are set to 0 and values above 1 are set to 1.<br> $$ Seasonal\:strength = 1 - \frac{Var(e_t)}{Var(s_{i,t}+e_t)} $$  |
| **Periods**             | `stl_features`              | Number of periods.                                                                                                                                                                                             |
| **Peak**                | `stl_features`              | Measures the strength of peaks by giving the maximum value in the seasonal component of the STL decomposition.                                                                                                 |
| **Trough**              | `stl_features`              | Measures the strength of troughs by giving the minimum value in the seasonal component of the STL decomposition.                                                                                               |
| **Heterogeneity**       | `heterogeneity`             | Measures **autoregressive conditional heteroskedasticity** (ARCH) effects after removing mean, trend, and AR information.<br> A GARCH(1,1) model is fitted to measure changes in conditional variance over time.       |
| **Entropy**             | `count_entropy` / `entropy` | Measures spectral entropy (Shannon's entropy), which indicates the forecastability of a time series.<br> Low values indicate a high signal-to-noise ratio; high values indicate lower forecastability.             |
| **Lumpiness**           | `lumpiness`                 | Measures the **variance of the variances across tiled**, non-overlapping windows.                                                                                                                                  |
| **Stability**           | `stability`                 | Measures the **variance of the means across tiled**, non-overlapping windows.                                                                                                                                      |
| **Sparsity**            | `sparsity`                  | Computes the average number of observations with zero values.                                                                                                                                                  |
| **Spikiness**           | `stl_features`              | Measures the strength of spikiness using trend and seasonality measures from STL decomposition.                                                                                                                |
| **Arch Model Features** | `arch_stat`                 | Computes a statistic based on the Lagrange Multiplier (LM) or ARCH to analyze effects left<br> unexplained by the econometric model.                                                                               |
| **Hurst**               | `hurst`                     | Measures long-term memory in a time series and indicates the level of fractional differencing.                                                                                                                 |
| **Hol Parameters**      | `holt_parameters`           | Returns the alpha (smoothing level) and beta (smoothing slope) parameters after fitting a **Holt linear trend**<br>model.                                                                                             |
| **Hw Parameters**       | `hw_parameters`             | Represents additional seasonal components: alpha, beta, and gamma.                                                                                                                                             |
| **Fspots**              | `flat_spots`                | Divides the sample space into ten equal-sized intervals and computes the maximum run length within any single interval.                                                                                        |
| **Cpoints**             | `crossing_points`           | Counts the number of times a time series crosses its median line.                                                                                                                                              |
| **Guerrero**            | `guerrero`                  | Selects the **lambda** that minimizes the **coefficient of variation** for **subseries of x**. It uses a variance-stabilizing transformation<br> and groups observations into subseries to estimate local means and variances. |

</div>

In [18]:
household_df = pd.read_parquet('./data/hourlyblock_file_7.parquet')
household_df.head()

,timestamp,LCLid,energy_consumption,frequency,series_length,stdorToU,Acorn,Acorn_grouped,file,holidays,...,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
0,2011-12-09 00:00:00+00:00,MAC000050,0.137,30 min,38976,Std,ACORN-D,Affluent,block_7,NO_HOLIDAY,...,267.0,7.27,1.55,1006.59,3.31,7.39,rain,clear-night,0.67,Clear
1,2011-12-09 00:30:00+00:00,MAC000050,0.113,30 min,38976,Std,ACORN-D,Affluent,block_7,NO_HOLIDAY,...,267.0,7.27,1.55,1006.59,3.31,7.39,None,None,0.67,None
2,2011-12-09 01:00:00+00:00,MAC000050,0.094,30 min,38976,Std,ACORN-D,Affluent,block_7,NO_HOLIDAY,...,261.0,6.43,1.67,1007.12,2.46,6.69,rain,clear-night,0.72,Clear
3,2011-12-09 01:30:00+00:00,MAC000050,0.066,30 min,38976,Std,ACORN-D,Affluent,block_7,NO_HOLIDAY,...,261.0,6.43,1.67,1007.12,2.46,6.69,None,None,0.72,None
4,2011-12-09 02:00:00+00:00,MAC000050,0.092,30 min,38976,Std,ACORN-D,Affluent,block_7,NO_HOLIDAY,...,257.0,6.29,0.45,1007.85,2.18,6.96,rain,clear-night,0.66,Clear


In [23]:
household_df[['precipType','icon','summary']] = household_df[['precipType','icon','summary']].ffill()

In [19]:
from src.imputation.interpolation import SeasonalInterpolation

household_ts = SeasonalInterpolation(seasonal_period=48*7).fit_transform(household_df.energy_consumption.values.reshape(-1,1)).squeeze()

household_df['energy_consumption'] = household_ts

In [27]:
mean_df = household_df.groupby("LCLid", as_index=False)["energy_consumption"].mean()
display(mean_df.sort_values(by="energy_consumption").head(10))

,LCLid,energy_consumption
47,MAC004814,0.039
2,MAC000073,0.085
17,MAC001370,0.087
14,MAC000776,0.092
16,MAC001135,0.094
26,MAC001826,0.115
6,MAC000289,0.137
25,MAC001825,0.145
13,MAC000741,0.155
7,MAC000317,0.165


In [30]:
hh_df_red = household_df[['LCLid','timestamp','energy_consumption']].rename(columns={'LCLid':'unique_id','timestamp':'ds','energy_consumption':'y'})

In [ ]:
all_features = [
    tsf.acf_features,
    tsf.arch_stat,
    tsf.crossing_points,
    tsf.entropy,
    tsf.flat_spots,
    tsf.heterogeneity,
    tsf.holt_parameters,
    tsf.lumpiness,
    tsf.nonlinearity,
    tsf.pacf_features,
    tsf.stl_features,
    tsf.stability,
    tsf.hw_parameters,
    tsf.unitroot_kpss,
    tsf.unitroot_pp,
    tsf.series_length,
    tsf.hurst,
]

all_feat = tsf.tsfeatures(hh_df_red, freq=4, features=all_features)
all_feat.head(10)